# 📰 News Category Dataset  
### JSON → CSV Conversion, Cleaning & Feature Engineering

🛒danload datasets:
be sure you danload News Category Dataset and 📊 News Category Classification Dataset

In [ ]:
import os
print(os.listdir("/kaggle/input"))


FileNotFoundError: [WinError 3] The system cannot find the path specified: '/News_Category_Dataset_v3.json'

## 📖 Introduction

In this notebook, we:
- Convert a large **JSONL** news dataset into CSV format
- Clean and normalize textual fields
- Engineer useful NLP features
- Prepare **ML-ready datasets** for text classification tasks

This dataset is suitable for:
- News category classification
- NLP preprocessing pipelines
- Feature engineering demonstrations


In [ ]:
# ============================================================
# Libraries
# ============================================================
from pandas import read_csv
from pandas import read_json
from pandas import to_datetime
from re import sub


df = read_json(
    r"D:\Projects\Kaggle\DataSet\test\News_Category_Dataset_v3.json",
    lines=True
)

print(df.shape)
df.to_csv("news_category.csv", index=False)

print("✅ JSONL converted to CSV successfully")


## 🔄 Why JSONL Conversion?

The original dataset is stored in **JSON Lines (JSONL)** format.
To simplify analysis and visualization, we convert it into a **CSV** file,
which is easier to manipulate using pandas and compatible with Kaggle tools.


## 🧹 Text Cleaning Strategy

We apply the following preprocessing steps:
- Lowercasing all text
- Removing URLs
- Removing non-alphabetic characters
- Normalizing whitespace

This ensures clean and consistent text for NLP tasks.


In [ ]:
# ============================================================
# Text Cleaning Function
# ============================================================
def clean_text(text):
    text = str(text).lower()
    text = sub(r"http\S+|www\S+", "", text)     # remove URLs
    text = sub(r"[^a-z\s]", "", text)           # keep letters only
    text = sub(r"\s+", " ", text).strip()
    return text


## 📂 Loading CSV Dataset

We now load the converted CSV file and inspect its original shape.


In [ ]:
# ============================================================
# Load CSV
# ============================================================
df = read_csv(r"D:\Projects\news_category.csv")

print("Original shape:", df.shape)


## 🎯 Column Selection

We keep only the most relevant columns for NLP and classification tasks.


In [ ]:
# ============================================================
# Keep useful columns
# ============================================================
df = df[[
    "headline",
    "short_description",
    "category",
    "authors",
    "date"
]]


## ✂️ Text Cleaning & Deduplication

- Clean headline and description text
- Remove very short articles
- Remove duplicate headlines


In [ ]:
# ============================================================
# Clean text columns
# ============================================================
df["clean_headline"] = df["headline"].apply(clean_text)
df["clean_description"] = df["short_description"].apply(clean_text)

df = df[df["clean_headline"].str.len() > 10]
df = df.drop_duplicates(subset="clean_headline")


## ⏰ Time Feature Engineering

Extracting year and month from publication date.


In [ ]:
# ============================================================
# Time features
# ============================================================
df["date"] = to_datetime(df["date"], errors="coerce")
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month


## 📊 Text Statistics

Basic NLP features extracted from headlines:
- Character count
- Word count
- Average word length


In [ ]:
# ============================================================
# Text statistics
# ============================================================
df["headline_char_count"] = df["clean_headline"].str.len()
df["headline_word_count"] = df["clean_headline"].str.split().str.len()
df["avg_word_length"] = (
    df["headline_char_count"] / df["headline_word_count"]
)


## 🧼 Final Cleanup

Removing unused columns and handling missing values.


In [ ]:
# ============================================================
# Final cleanup
# ============================================================
df = df.drop(columns=["headline", "short_description"])
df = df.fillna("Unknown")

print("Cleaned shape:", df.shape)
print("\nCategory distribution (%):")
print(df["category"].value_counts(normalize=True) * 100)


## 💾 Saving Outputs

We save:
- A fully cleaned dataset
- A machine-learning-ready subset


In [ ]:
# ============================================================
# Save cleaned datasets
# ============================================================
df.to_csv("news_category_clean.csv", index=False)

df_ml = df[[
    "clean_headline",
    "category",
    "headline_word_count",
    "avg_word_length",
    "year"
]]

df_ml.to_csv("news_category_ml_ready.csv", index=False)

print("\n✅ Cleaned CSV files saved successfully.")


# 🎉Final Words
I hope this notebook was useful for you. If you liked this notebook, I would be very happy if you would give it an upvote. thanks all programmers